In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
import networkx as nx
import os
import pandas as pd
import torch_geometric as torch_g
from torch_geometric.nn import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from pathlib import Path

In [2]:
# --- 0. Pre-computation of Node2Vec Tissue Embeddings & Hierarchical Pairs ---
# This block is similar to before, ensuring all necessary components are ready.

data_dir = Path('./Data')

# Load the tissue hierarchy graph
G_tissue = nx.read_edgelist(data_dir / 'tissue.hierarchy', create_using=nx.DiGraph())
tissue_nodes = sorted(list(G_tissue.nodes())) # Sort for consistent ordering
node_to_idx_tissue = {node: i for i, node in enumerate(tissue_nodes)}
edges_tissue = [(node_to_idx_tissue[src], node_to_idx_tissue[dst]) for src, dst in G_tissue.edges()]
edge_index_tissue = torch.tensor(edges_tissue, dtype=torch.long).t().contiguous()

# Initialize Node2Vec (embedding_dim=128 as in Appendix I diagram)
# Ensure num_nodes matches len(tissue_nodes)
node2vec_engine = Node2Vec(
    edge_index_tissue,
    embedding_dim=256,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    p=1, q=1,
    sparse=True,
    num_nodes=len(tissue_nodes) # Explicitly pass num_nodes
)

# Ensure the Node2Vec model is on the correct device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Added Xavier Initialization for stable training at high dimensions
torch.nn.init.xavier_normal_(node2vec_engine.embedding.weight)
node2vec_engine = node2vec_engine.to(device)

print("Training Node2Vec embeddings...")
loader = node2vec_engine.loader(batch_size=128, shuffle=True, num_workers=0)
optimizer_n2v = torch.optim.SparseAdam(list(node2vec_engine.parameters()), lr=0.01)

node2vec_epochs = 1000
for epoch in range(1, node2vec_epochs + 1):
    node2vec_engine.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer_n2v.zero_grad()
        loss_n2v = node2vec_engine.loss(pos_rw.to(device), neg_rw.to(device))
        loss_n2v.backward()
        optimizer_n2v.step()
        total_loss += loss_n2v.item()
        
        # --- Memory Management ---
        del pos_rw, neg_rw, loss_n2v
            
    if epoch % 100 == 0:
        print(f"  Node2Vec Epoch: {epoch:03d}, Loss: {total_loss / len(loader):.4f}")
print("Node2Vec training complete.")

# Move embeddings back to cpu before taking their raw weights
node2vec_engine = node2vec_engine.to('cpu')

tissue_embeddings_raw = node2vec_engine.embedding.weight.data

if 'Root_NODE' in node_to_idx_tissue:
    root_node_idx = node_to_idx_tissue['Root_NODE']
    global_tissue_address = tissue_embeddings_raw[root_node_idx].unsqueeze(0)
else:
    global_tissue_address = tissue_embeddings_raw.mean(dim=0, keepdim=True)
global_tissue_address = global_tissue_address.to('cuda' if torch.cuda.is_available() else 'cpu')


# Prepare Hierarchical Pairs for Loss Function
# Correctly identify child and parent from the hierarchy edges (child -> parent)
hierarchical_pairs_list = []
for child_name, parent_name in G_tissue.edges():
    child_idx = node_to_idx_tissue[child_name]
    parent_idx = node_to_idx_tissue[parent_name]
    hierarchical_pairs_list.append((child_idx, parent_idx))
hierarchical_pairs = torch.tensor(hierarchical_pairs_list, dtype=torch.long)

Training Node2Vec embeddings...
  Node2Vec Epoch: 100, Loss: 0.7672
  Node2Vec Epoch: 200, Loss: 0.7549
  Node2Vec Epoch: 300, Loss: 0.7519
  Node2Vec Epoch: 400, Loss: 0.7513
  Node2Vec Epoch: 500, Loss: 0.7486
  Node2Vec Epoch: 600, Loss: 0.7491
  Node2Vec Epoch: 700, Loss: 0.7466
  Node2Vec Epoch: 800, Loss: 0.7476
  Node2Vec Epoch: 900, Loss: 0.7472
  Node2Vec Epoch: 1000, Loss: 0.7474
Node2Vec training complete.


In [3]:
# --- 1. Define the LateFusionGAT Model (Same as before) ---
class LateFusionGAT(nn.Module):
    def __init__(self, num_proteins, protein_embedding_dim,
                 num_heads, gat_hidden_channels, gat_output_channels,
                 tissue_address_dim, mlp_hidden_channels, num_tissues,
                 global_tissue_address_tensor, dropout_rate=0.3):
        super(LateFusionGAT, self).__init__()

        self.protein_embedding = nn.Embedding(num_proteins, protein_embedding_dim)

        self.gat1 = GATConv(
            in_channels=protein_embedding_dim,
            out_channels=gat_hidden_channels,
            heads=num_heads,
            dropout=dropout_rate,
            add_self_loops=False,
            negative_slope=0.2
        )
        self.gat2 = GATConv(
            in_channels=gat_hidden_channels * num_heads,
            out_channels=gat_output_channels,
            heads=1,
            dropout=dropout_rate,
            add_self_loops=False
        )
        
        self.global_tissue_address = nn.Parameter(global_tissue_address_tensor, requires_grad=False)
        
        mlp_input_dim = (gat_output_channels * 2) + tissue_address_dim

        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, mlp_hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_hidden_channels, mlp_hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_hidden_channels // 2, num_tissues)
        )

    def forward(self, batch):
        x = batch.x
        edge_index = batch.edge_index
        
        # Note: LinkNeighborLoader automatically remaps edge_label_index 
        # to local indices relative to the sampled batch (x).
        protein_a_indices = batch.edge_label_index[0]
        protein_b_indices = batch.edge_label_index[1]

        x_embedded = self.protein_embedding(x)

        x_gat = F.elu(self.gat1(x_embedded, edge_index))
        x_gat = self.gat2(x_gat, edge_index)

        # Directly index using the remapped edge_label_index
        emb_protein_a = x_gat[protein_a_indices]
        emb_protein_b = x_gat[protein_b_indices]

        batch_size = emb_protein_a.shape[0]
        repeated_tissue_address = self.global_tissue_address.repeat(batch_size, 1)

        concatenated_features = torch.cat(
            (emb_protein_a, emb_protein_b, repeated_tissue_address),
            dim=-1
        )

        raw_predictions = self.mlp(concatenated_features)
        probabilities = torch.sigmoid(raw_predictions)
        
        return probabilities


In [4]:
# --- 2. Custom Loss Function with Normalized Penalty ---
def custom_loss(y_hat, y_true, hierarchical_pairs_tensor, lambda_penalty=0.1):
    bce_loss = F.binary_cross_entropy(y_hat, y_true)
    
    children_probs = y_hat[:, hierarchical_pairs_tensor[:, 0]]
    parents_probs = y_hat[:, hierarchical_pairs_tensor[:, 1]]
    
    # We use mean() instead of sum() to make penalty independent of batch size
    violations = torch.clamp(children_probs - parents_probs, min=0.0)
    penalty = lambda_penalty * torch.mean(violations)
    
    return bce_loss + penalty

In [5]:
print("Setting up data pipeline...")
ppi_file_path = data_dir / 'PPT-Ohmnet_tissues-combined.edgelist'
df = pd.read_csv(ppi_file_path, sep='\t', header=0, names=['protein_a', 'protein_b', 'tissue'])

unique_proteins = pd.concat([df['protein_a'], df['protein_b']]).unique()
protein_to_idx = {prot: i for i, prot in enumerate(unique_proteins)}
num_unique_proteins = len(unique_proteins)

# Use the same 'tissue_nodes' list from Node2Vec setup for consistency
num_tissues = len(tissue_nodes) # This is 144 (or 145 if Root_NODE is included)

df['p1_idx'] = df['protein_a'].map(protein_to_idx)
df['p2_idx'] = df['protein_b'].map(protein_to_idx)
# Map tissue names from PPI data to the *same* indices used by Node2Vec

# --- Fix: Robust Tissue Mapping (Name -> BTO ID -> Local Index) ---
print("Building Name-to-BTO mapping from BrendaTissue.obo...")
obo_path = data_dir / 'BrendaTissue.obo'
name_to_bto = {}
if obo_path.exists():
    with open(obo_path, 'r', encoding='utf-8') as f:
        cid = None
        for line in f:
            line = line.strip()
            if line.startswith('id: '): cid = line[4:]
            elif line.startswith('name: ') and cid: name_to_bto[line[6:].lower()] = cid

def map_tissue(t_name):
    # Normalize: 'urinary_bladder' -> 'urinary bladder'
    search_name = t_name.lower().replace('_', ' ')
    bto_id = name_to_bto.get(search_name)
    if not bto_id: return None
    # Check hierarchy nodes (some have _NODE suffix)
    if bto_id in node_to_idx_tissue: return node_to_idx_tissue[bto_id]
    if bto_id + '_NODE' in node_to_idx_tissue: return node_to_idx_tissue[bto_id + '_NODE']
    return None

df['t_idx'] = df['tissue'].map(map_tissue)
old_len = len(df)
df = df.dropna(subset=['t_idx'])
df['t_idx'] = df['t_idx'].astype(int)
print(f"Mapped {len(df)}/{old_len} PPI records to hierarchy node indices.")

if len(df) == 0:
    print("ERROR: Mapping failed completely! Check tissue names in PPI file vs OBO.")

# Use the mapped indices to group
grouped = df.groupby(['p1_idx', 'p2_idx'])['t_idx'].apply(list).reset_index()

source_nodes_all = torch.tensor(grouped['p1_idx'].values, dtype=torch.long)
target_nodes_all = torch.tensor(grouped['p2_idx'].values, dtype=torch.long)
edge_index_all_ppi = torch.stack([source_nodes_all, target_nodes_all], dim=0)

y_targets_all_ppi = torch.zeros((len(grouped), num_tissues), dtype=torch.float)
for row_idx, tissues in enumerate(grouped['t_idx']):
    # Ensure we don't try to set a target for a tissue index that's out of bounds
    # (e.g., if 'Root_NODE' is in tissue_nodes but not in the PPI targets)
    y_targets_all_ppi[row_idx, tissues] = 1.0

protein_indices_all = torch.arange(num_unique_proteins, dtype=torch.long)

# The full graph data object (used for neighbor sampling, not direct training)
full_graph_data = Data(
    x=protein_indices_all,
    edge_index=edge_index_all_ppi # Full PPI graph structure for sampling
)

# Split the *unique PPIs* (edges for prediction) and their labels (y_targets)
# We need to split the *indices* of the grouped DataFrame
num_unique_ppis = len(grouped)
all_indices = torch.arange(num_unique_ppis, dtype=torch.long)

train_indices, test_val_indices = train_test_split(all_indices, test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(test_val_indices, test_size=0.5, random_state=42) # 0.5 of 20% = 10%

print(f"Total unique PPIs: {num_unique_ppis}")
print(f"Train PPIs: {len(train_indices)}")
print(f"Validation PPIs: {len(val_indices)}")
print(f"Test PPIs: {len(test_indices)}")

# Create LinkNeighborLoaders for each split
train_loader = LinkNeighborLoader(
    full_graph_data,
    num_neighbors=[15, 10],
    batch_size=1024,
    edge_label_index=edge_index_all_ppi[:, train_indices], # Only sample edges from training split
    edge_label=y_targets_all_ppi[train_indices],           # Corresponding labels
    shuffle=True,
)

val_loader = LinkNeighborLoader(
    full_graph_data,
    num_neighbors=[15, 10],
    batch_size=1024, # Can use a larger batch size for validation if VRAM allows
    edge_label_index=edge_index_all_ppi[:, val_indices],
    edge_label=y_targets_all_ppi[val_indices],
    shuffle=False, # No need to shuffle validation data
)

test_loader = LinkNeighborLoader(
    full_graph_data,
    num_neighbors=[15, 10],
    batch_size=1024, # Can use a larger batch size for testing
    edge_label_index=edge_index_all_ppi[:, test_indices],
    edge_label=y_targets_all_ppi[test_indices],
    shuffle=False,
)
print("Train, Validation, Test loaders ready.")

# --- Identify Leaf Node Indices for Test Set Masking ---
# Identified 107 leaf nodes using in_degree (most specific tissues in child->parent graph)
# Identified 107 leaf nodes using in_degree (fixing the out_degree bug)
leaf_node_names = [node for node in G_tissue.nodes() if G_tissue.in_degree(node) == 0 and node != 'Root_NODE']
leaf_node_indices = torch.tensor([node_to_idx_tissue[name] for name in leaf_node_names if name in node_to_idx_tissue], dtype=torch.long)
print(f"Identified {len(leaf_node_indices)} leaf nodes for test set masking.")


# --- Model Instantiation ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Setting up data pipeline...
Building Name-to-BTO mapping from BrendaTissue.obo...
Mapped 3597190/3666563 PPI records to hierarchy node indices.
Total unique PPIs: 70338
Train PPIs: 56270
Validation PPIs: 7034
Test PPIs: 7034
Train, Validation, Test loaders ready.
Identified 107 leaf nodes for test set masking.
Using device: cuda


In [ ]:
# --- 3. Data Loading & Train/Validation/Test Split ---
if __name__ == "__main__":


    protein_embedding_dim = 256
    num_heads = 16
    gat_hidden_channels = 256
    gat_output_channels = 256
    tissue_address_dim = global_tissue_address.shape[1]
    mlp_hidden_channels = 1024
    dropout_rate = 0.3

    model = LateFusionGAT(
        num_proteins=num_unique_proteins,
        protein_embedding_dim=protein_embedding_dim,
        num_heads=num_heads,
        gat_hidden_channels=gat_hidden_channels,
        gat_output_channels=gat_output_channels,
        tissue_address_dim=tissue_address_dim,
        mlp_hidden_channels=mlp_hidden_channels,
        num_tissues=num_tissues, # Use the consistent num_tissues from Node2Vec setup
        global_tissue_address_tensor=global_tissue_address,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # --- Training and Validation Loop ---
    num_epochs = 1000
    patience = 50
    early_stop_counter = 0
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        for i, batch in enumerate(train_loader):
            batch = batch.to(device)
            optimizer.zero_grad()
            
            predictions = model(batch)
            loss = custom_loss(predictions, batch.edge_label, hierarchical_pairs.to(device), lambda_penalty=0.1)
            
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f"  Epoch {epoch+1}, Batch {i+1}, Train Loss: {loss.item():.4f}")
                
            # --- Memory Management ---
        # Clean up after epoch
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"\nEpoch {epoch+1} finished. Avg Train Loss: {avg_train_loss:.4f}")

        # --- Validation Loop ---
        model.eval()
        total_val_loss = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                batch = batch.to(device)
                predictions = model(batch)
                loss = custom_loss(predictions, batch.edge_label, hierarchical_pairs.to(device), lambda_penalty=0.1)
                total_val_loss += loss.item()

                all_val_preds.append(predictions.cpu())
                all_val_labels.append(batch.edge_label.cpu())
                
                # --- Memory Management ---
        avg_val_loss = total_val_loss / len(val_loader)
        print(f"  Validation Loss: {avg_val_loss:.4f}")

        # Concatenate all predictions and labels for metric calculation
        all_val_preds = torch.cat(all_val_preds, dim=0)
        all_val_labels = torch.cat(all_val_labels, dim=0)

        # Calculate ROCAUC, ROCPRC, Macro-F1 (for validation on all 144 tissues)
        # Note: These are for multi-label classification.
        # Ensure that `y_true` and `y_score` are 1D arrays for each metric calculation.
        # This can be done by flattening or iterating over each column (tissue).

        val_roc_auc = []
        y_true_np = all_val_labels.numpy(); y_pred_np = all_val_preds.numpy()
        # Compute metrics on CPU with NumPy for stability
        y_true_np = all_val_labels.numpy()
        y_pred_np = all_val_preds.numpy()
        val_roc_prc = []
        val_f1 = []

        for j in range(num_tissues):
            # Evaluate only if both positive and negative classes are present
            if y_true_np[:, j].sum() > 0 and (1.0 - y_true_np[:, j]).sum() > 0:
                try:
                    val_roc_auc.append(roc_auc_score(y_true_np[:, j], y_pred_np[:, j]))
                    val_roc_prc.append(average_precision_score(y_true_np[:, j], y_pred_np[:, j]))
                    # For F1, you typically need to binarize predictions. Let's use a common threshold of 0.5 for now.
                    val_f1.append(f1_score(y_true_np[:, j], (y_pred_np[:, j] > 0.5).astype(float)))
                except ValueError:
                    # Handle cases where roc_auc_score might fail (e.g., only one class in y_true)
                    pass
        
        avg_val_roc_auc = sum(val_roc_auc) / len(val_roc_auc) if val_roc_auc else 0
        avg_val_roc_prc = sum(val_roc_prc) / len(val_roc_prc) if val_roc_prc else 0
        avg_val_f1 = sum(val_f1) / len(val_f1) if val_f1 else 0

        print(f"  Validation ROCAUC: {avg_val_roc_auc:.4f}, ROCPRC: {avg_val_roc_prc:.4f}, Macro-F1: {avg_val_f1:.4f}")

        # Save best model based on validation loss or ROCAUC
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0 # Reset counter
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss
            }, 'best_model.pt')
            print("  >>> New best model checkpoint saved!")
        else:
            early_stop_counter += 1
            
        # Regular checkpointing every 50 epochs
        if (epoch + 1) % 50 == 0:
            os.makedirs('checkpoints', exist_ok=True)
            checkpoint_path = f'checkpoints/checkpoint_epoch_{epoch+1}.pt'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': avg_val_loss
            }, checkpoint_path)
            print(f"  Saved epoch checkpoint to {checkpoint_path}")
        if early_stop_counter >= patience:
            print(f'Early stopping triggered after {epoch+1} epochs.')
            break


    # --- Test Loop (After training is complete) ---
    print("\n--- Starting Final Test Evaluation ---")
    checkpoint = torch.load('best_model.pt', weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict']) # Load the best model
    model.eval()
    total_test_loss = 0
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            batch = batch.to(device)
            predictions = model(batch)
            loss = custom_loss(predictions, batch.edge_label, hierarchical_pairs.to(device), lambda_penalty=0.1)
            total_test_loss += loss.item()

            all_test_preds.append(predictions.cpu())
            all_test_labels.append(batch.edge_label.cpu())
            
            # --- Memory Management ---
    avg_test_loss = total_test_loss / len(test_loader)
    print(f"  Test Loss: {avg_test_loss:.4f}")

    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    # --- Final Test Metrics ---
    y_true_test = all_test_labels.numpy()
    y_pred_test = all_test_preds.numpy()
    
    # Metrics for ALL tissues
    all_roc_auc = [roc_auc_score(y_true_test[:, j], y_pred_test[:, j]) for j in range(num_tissues) if y_true_test[:, j].sum() > 0 and (1.0 - y_true_test[:, j]).sum() > 0]
    print(f'  Final Test ROCAUC (All 144 Tissues): {sum(all_roc_auc)/len(all_roc_auc):.4f}')
    
    # Metrics for LEAF tissues (Goal target)
    # Select only the predictions and labels for the 107 leaf nodes
    test_preds_leaf = all_test_preds[:, leaf_node_indices].numpy()
    test_labels_leaf = all_test_labels[:, leaf_node_indices].numpy()
    test_roc_auc = []
    for j in range(len(leaf_node_indices)):
        if test_labels_leaf[:, j].sum() > 0 and (1.0 - test_labels_leaf[:, j]).sum() > 0:
            try:
                test_roc_auc.append(roc_auc_score(test_labels_leaf[:, j], test_preds_leaf[:, j]))
            except: pass
    avg_test_roc_auc = sum(test_roc_auc) / len(test_roc_auc) if test_roc_auc else 0
    print(f'  Final Test ROCAUC (105 Leaf Tissues): {avg_test_roc_auc:.4f}')
    target_roc_auc = 0.756
    if avg_test_roc_auc > target_roc_auc:
        print(f"  Goal Achieved! Test ROCAUC ({avg_test_roc_auc:.4f}) exceeded target ({target_roc_auc:.4f}).")
    else:
        print(f"  Goal Not Met. Test ROCAUC ({avg_test_roc_auc:.4f}) is below target ({target_roc_auc:.4f}).")

Setting up data pipeline...
Building Name-to-BTO mapping from BrendaTissue.obo...
Mapped 3597190/3666563 PPI records to hierarchy node indices.
Total unique PPIs: 70338
Train PPIs: 56270
Validation PPIs: 7034
Test PPIs: 7034
Train, Validation, Test loaders ready.
Identified 107 leaf nodes for test set masking.
Using device: cuda

Epoch 1 finished. Avg Train Loss: 0.4054
  Validation Loss: 0.3717
  Validation ROCAUC: 0.6218, ROCPRC: 0.4530, Macro-F1: 0.2476
  >>> New best model checkpoint saved!

Epoch 2 finished. Avg Train Loss: 0.3708
  Validation Loss: 0.3619
  Validation ROCAUC: 0.6559, ROCPRC: 0.4777, Macro-F1: 0.2817
  >>> New best model checkpoint saved!

Epoch 3 finished. Avg Train Loss: 0.3619
  Validation Loss: 0.3490
  Validation ROCAUC: 0.6929, ROCPRC: 0.5118, Macro-F1: 0.3461
  >>> New best model checkpoint saved!

Epoch 4 finished. Avg Train Loss: 0.3556
  Validation Loss: 0.3426
  Validation ROCAUC: 0.7136, ROCPRC: 0.5241, Macro-F1: 0.3324
  >>> New best model checkpoint 

In [ ]:
# --- Automatically Execute Setup Cells ---
# Run this once you open the notebook to avoid scrolling up and running cells 1 through 4 manually.
# It executes imports, Node2Vec embedding prep, the GAT class definition, and the data pipeline.

import json
import sys

print("Executing setup cells... (This might take a moment due to Node2Vec)")

notebook_path = 'with validation.ipynb'
try:
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb_data = json.load(f)
    
    # We execute indices: 0 (Imports), 1 (Node2Vec), 2 (Model), 3 (Loss), 5 (Data Pipeline)
    # This prepares the environment perfectly for loading the saved weights.
    setup_indices = [0, 1, 2, 3, 4]
    
    for idx in setup_indices:
        if idx < len(nb_data['cells']):
            source_code = "".join(nb_data['cells'][idx].get('source', []))
            if source_code.strip():
                print(f"\n--- Executing Cell index {idx} ---")
                # Using globals() ensures that everything loaded inside here becomes
                # available to all subsequent cells, just as if you ran them natively.
                exec(source_code, globals())
                
    print("\nSetup cells executed successfully! You can now load the model.")
except Exception as e:
    print(f"\nError occurred while executing setup cells: {e}")

Executing setup cells... (This might take a moment due to Node2Vec)

--- Executing Cell index 0 ---

--- Executing Cell index 1 ---
Training Node2Vec embeddings...
  Node2Vec Epoch: 100, Loss: 0.7688
  Node2Vec Epoch: 200, Loss: 0.7581
  Node2Vec Epoch: 300, Loss: 0.7522
  Node2Vec Epoch: 400, Loss: 0.7505
  Node2Vec Epoch: 500, Loss: 0.7498
  Node2Vec Epoch: 600, Loss: 0.7510
  Node2Vec Epoch: 700, Loss: 0.7467
  Node2Vec Epoch: 800, Loss: 0.7476
  Node2Vec Epoch: 900, Loss: 0.7463
  Node2Vec Epoch: 1000, Loss: 0.7433
Node2Vec training complete.

--- Executing Cell index 2 ---

--- Executing Cell index 3 ---

--- Executing Cell index 5 ---
Setting up data pipeline...
Building Name-to-BTO mapping from BrendaTissue.obo...
Mapped 3597190/3666563 PPI records to hierarchy node indices.
Total unique PPIs: 70338
Train PPIs: 56270
Validation PPIs: 7034
Test PPIs: 7034
Train, Validation, Test loaders ready.
Identified 107 leaf nodes for test set masking.
Using device: cuda

Setup cells execute

In [6]:
# Cell to load the best model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate the model architecture
best_model = LateFusionGAT(
    num_proteins=num_unique_proteins,
    protein_embedding_dim=256,
    num_heads=16,
    gat_hidden_channels=256,
    gat_output_channels=256,
    tissue_address_dim=global_tissue_address.shape[1],
    mlp_hidden_channels=1024,
    num_tissues=num_tissues,
    global_tissue_address_tensor=global_tissue_address,
    dropout_rate=0.3
).to(device)

# Load the saved best model weights
checkpoint = torch.load('best_model.pt', map_location=device, weights_only=True)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()
print("Best model loaded successfully!")

Best model loaded successfully!


In [7]:
# Cell to calculate the test ROCAUC, ROCPRC and MACRO-F1 scores
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import numpy as np

all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        predictions = best_model(batch)
        all_test_preds.append(predictions.cpu())
        all_test_labels.append(batch.edge_label.cpu())

all_test_preds = torch.cat(all_test_preds, dim=0).numpy()
all_test_labels = torch.cat(all_test_labels, dim=0).numpy()

test_roc_auc = []
test_roc_prc = []
test_macro_f1 = []

for j in range(num_tissues):
    # Evaluate only if both positive and negative classes are present
    if all_test_labels[:, j].sum() > 0 and (1.0 - all_test_labels[:, j]).sum() > 0:
        try:
            test_roc_auc.append(roc_auc_score(all_test_labels[:, j], all_test_preds[:, j]))
            test_roc_prc.append(average_precision_score(all_test_labels[:, j], all_test_preds[:, j]))
            test_macro_f1.append(f1_score(all_test_labels[:, j], (all_test_preds[:, j] > 0.5).astype(float)))
        except ValueError:
            pass

avg_test_roc_auc = np.mean(test_roc_auc) if test_roc_auc else 0
avg_test_roc_prc = np.mean(test_roc_prc) if test_roc_prc else 0
avg_test_macro_f1 = np.mean(test_macro_f1) if test_macro_f1 else 0

print("Final Test Evaluation Metrics (Averaged over valid tissues):")
print(f"  Test ROCAUC:   {avg_test_roc_auc:.4f}")
print(f"  Test ROCPRC:   {avg_test_roc_prc:.4f}")
print(f"  Test MACRO-F1: {avg_test_macro_f1:.4f}")

Final Test Evaluation Metrics (Averaged over valid tissues):
  Test ROCAUC:   0.9831
  Test ROCPRC:   0.9184
  Test MACRO-F1: 0.8625


In [9]:
# Cell for attention analysis
print("Attention analysis structure preparation...")

# 1. Provide mapping from integer node IDs back to meaningful protein names:
if 'protein_to_idx' in globals():
    idx_to_protein = {v: k for k, v in protein_to_idx.items()}
else:
    print("Warning: protein_to_idx not found. Fallback down to index numbers...")
    idx_to_protein = {}

# 2. Extract attention from model's inner GAT layers
def get_attention_weights(model, batch):
    x = batch.x
    edge_index = batch.edge_index
    
    x_embedded = model.protein_embedding(x)
    
    # We call gat1 and gat2 with return_attention_weights=True to intercept the attention logits
    x_gat, (edge_index1, attn_weights1) = model.gat1(x_embedded, edge_index, return_attention_weights=True)
    x_gat = F.elu(x_gat)
    x_gat, (edge_index2, attn_weights2) = model.gat2(x_gat, edge_index, return_attention_weights=True)
    
    return attn_weights1, attn_weights2, edge_index1, edge_index2

# Retrieve one sample subgraph batch from the test_loader
sample_batch = next(iter(test_loader)).to(device)
with torch.no_grad():
    attn1, attn2, e1, e2 = get_attention_weights(best_model, sample_batch)

# The edges e2 refer to indices within the sample_batch.x.
# We must map them back to global indices to correctly fetch protein IDs.
e2_src_global = sample_batch.x[e2[0]]
e2_dst_global = sample_batch.x[e2[1]]

print(f"Layer 1 Attention Weights Shape: {attn1.shape}")
print(f"Layer 2 Attention Weights Shape: {attn2.shape} (Average across heads)")
print(f"Edges analyzed in layer 1: {e1.shape[1]}")
print(f"Edges analyzed in layer 2: {e2.shape[1]}")

# Process attention weights
if attn2.dim() == 2:
    mean_attn2 = attn2.mean(dim=-1) # average multi-head attention
else:
    mean_attn2 = attn2

# Find the edges with highest attention
top_k_edges = min(15, mean_attn2.shape[0])
top_attn_vals, top_attn_idx = torch.topk(mean_attn2, top_k_edges)

print(f"\n--- Top {top_k_edges} Most Attended Protein-Protein Interactions in Layer 2 ---")
for i in range(top_k_edges):
    idx = top_attn_idx[i]
    src_prot_idx = e2_src_global[idx].item()
    dst_prot_idx = e2_dst_global[idx].item()
    
    src_prot_name = idx_to_protein.get(src_prot_idx, f"Protein_{src_prot_idx}")
    dst_prot_name = idx_to_protein.get(dst_prot_idx, f"Protein_{dst_prot_idx}")
    
    val = top_attn_vals[i].item()
    
    # Visual Arrow Representation using extracted protein names instead of obscure integers
    print(f"  {src_prot_name} --> {dst_prot_name} | Attention Weight: {val:.4f}")

# Give a quick statistical breakdown to help interpret the strength of the weights
print(f"\n--- Attention weight distribution in sample (Layer 2) ---")
print(f"  Max:  {mean_attn2.max().item():.4f}")
print(f"  Mean: {mean_attn2.mean().item():.4f}")
print(f"  Min:  {mean_attn2.min().item():.4f}")

Attention analysis structure preparation...
Layer 1 Attention Weights Shape: torch.Size([28373, 16])
Layer 2 Attention Weights Shape: torch.Size([28373, 1]) (Average across heads)
Edges analyzed in layer 1: 28373
Edges analyzed in layer 2: 28373

--- Top 15 Most Attended Protein-Protein Interactions in Layer 2 ---
  5454 --> 5454 | Attention Weight: 1.0000
  26270 --> 80020 | Attention Weight: 1.0000
  7059 --> 7059 | Attention Weight: 1.0000
  8914 --> 54962 | Attention Weight: 1.0000
  23405 --> 114803 | Attention Weight: 1.0000
  10482 --> 4983 | Attention Weight: 1.0000
  3064 --> 11221 | Attention Weight: 1.0000
  5578 --> 56302 | Attention Weight: 1.0000
  4343 --> 23025 | Attention Weight: 1.0000
  81628 --> 65267 | Attention Weight: 1.0000
  6310 --> 1139 | Attention Weight: 1.0000
  2885 --> 57007 | Attention Weight: 1.0000
  2033 --> 2056 | Attention Weight: 1.0000
  84100 --> 84100 | Attention Weight: 1.0000
  5341 --> 5341 | Attention Weight: 1.0000

--- Attention weight di